# 第 2 節｜BERT 二元分類微調

| 項目 | 重點 |
|---|---|
| 資料 | `reviews.txt`：飯店評論（UTF-8、Tab 分隔、無標題列） |
| 欄位 | `sentence` 評論、`label` 標籤（0 負面／1 正面） |
| 筆數 | 有效 **7,765** 筆（正面 5,322／負面 2,443） |
| 資料切分 | Train 4,969／Validation 1,243／Test 1,553 |
| 訓練 | Epoch 上限＋固定 Steps 驗證／儲存 Checkpoint |
| 最佳模型 | `03_bert_reviews_output/best_model/` |

**操作順序：**上傳檔案 → 切分資料 → 微調模型 → 查看 Loss → Test → 新評論預測。


## 0｜準備資料

- Colab：選擇 **T4 GPU**。
- 上傳 **`reviews.txt`** 到 `/content/`。
- 從下一格依序執行。


In [ ]:
# Install the packages required for BERT fine-tuning.
# 安裝 BERT 微調需要的套件。
%pip -q install transformers==4.57.1 datasets==3.6.0 accelerate sentencepiece scikit-learn

# Import the path utility for locating the uploaded dataset.
# 匯入檔案路徑工具，以確認資料集已上傳。
from pathlib import Path

# Import tools for reading data and drawing learning curves.
# 匯入資料讀取與訓練曲線繪圖工具。
import pandas as pd

# Import the plotting library.
# 匯入繪製 Loss 曲線的函式庫。
import matplotlib.pyplot as plt

# Import the framework used for model training.
# 匯入模型訓練使用的 PyTorch。
import torch

# Import the data conversion utility.
# 匯入 Hugging Face 的資料集轉換工具。
from datasets import Dataset

# Import stratified data splitting and evaluation metrics.
# 匯入分層資料切分及分類評估工具。
from sklearn.model_selection import train_test_split

# Import the evaluation helpers.
# 匯入分類評估指標。
from sklearn.metrics import accuracy_score, classification_report

# Import the tokenizer, classifier, and training tools.
# 匯入 Tokenizer、分類模型與微調工具。
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments, pipeline

# Specify the uploaded tab-separated review file.
# 指定上傳的 Tab 分隔評論檔案。
DATA_FILE = Path("reviews.txt")

# Check the file before attempting to read it.
# 讀取前先確認檔案存在，避免看不懂的路徑錯誤。
if not DATA_FILE.is_file():
    # Explain how to fix a missing upload.
    # 提示尚未上傳時該如何處理。
    raise FileNotFoundError("找不到 reviews.txt；請先將檔案上傳至目前 Notebook 工作目錄。")

# Read the two unnamed columns separated by a tab.
# 以 Tab 分隔讀取沒有標題列的「評論文字、標籤」兩欄。
df = pd.read_csv(DATA_FILE, sep="\t", header=None, names=["sentence", "label"], dtype={"sentence": "string", "label": "Int64"}, encoding="utf-8")

# Count the original rows before removing invalid data.
# 清理前先記錄原始資料的行數。
original_count = len(df)

# Remove rows without either text or a label.
# 移除缺少評論或標籤的資料列。
df = df.dropna(subset=["sentence", "label"]).copy()

# Remove surrounding whitespace from each review.
# 清除評論前後多餘的空白字元。
df["sentence"] = df["sentence"].str.strip()

# Exclude reviews whose text is empty after trimming.
# 排除去除空白後沒有實際文字的評論。
df = df.loc[df["sentence"].ne("")].copy()

# Verify that the binary labels match the lesson definition.
# 檢查資料標籤是否符合課程的 0／1 二元分類定義。
if not df["label"].isin([0, 1]).all():
    # Stop rather than silently changing unexpected labels.
    # 遇到其他標籤時停止，避免擅自更動原始標註。
    raise ValueError("資料的 label 必須只有 0（負面）或 1（正面）。")

# Convert the verified labels to plain integers.
# 將已確認的標籤轉成一般整數。
df["label"] = df["label"].astype(int)

# Remove identical reviews to avoid cross-partition leakage.
# 移除重複評論，避免相同文字同時出現在不同資料組。
df = df.drop_duplicates(subset=["sentence"]).reset_index(drop=True)

# Use all valid reviews by default; set a number for a quicker trial.
# 預設使用所有有效評論；可設成數字進行較快的小規模測試。
SAMPLE_SIZE = None

# Select a reproducible subset only if requested.
# 只有指定抽樣筆數時才使用固定亂數抽取子集。
if SAMPLE_SIZE is not None:
    # Draw at most the available number of reviews.
    # 抽取筆數不得超過現有的有效評論筆數。
    df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)

# Choose the pretrained Chinese BERT checkpoint.
# 指定要微調的預訓練中文 BERT 模型。
MODEL_ID = "google-bert/bert-base-chinese"

# Display data size, cleaning result, and label imbalance.
# 顯示資料量、清理結果與正負面數量差異。
print("原始行數：", original_count, "｜清理後本次使用：", len(df))

# Display class counts before splitting.
# 切分前顯示各類別的資料筆數。
print("負面 0／正面 1：", df["label"].value_counts().sort_index().to_dict())

## 1｜切分資料與 Tokenizer

| 資料集 | 比例 | 預設筆數 | 用途 |
|---|---:|---:|---|
| Train | 64% | 4,969 | 訓練模型 |
| Validation | 16% | 1,243 | 選最佳模型、Early Stopping |
| Test | 20% | 1,553 | 最後評估 |

**可調設定：**`SAMPLE_SIZE = None` 使用全部資料；改成 `5000` 可縮短訓練時間。切分後筆數會隨之改變。


In [ ]:
# Reserve 20% of the reviews for the final test, keeping class proportions.
# 先保留 20% 作為最終測試資料，維持正負面標籤比例。
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# Reserve 20% of the remaining reviews for validation.
# 再從剩餘資料保留 20% 作為驗證集（即原始資料的 16%）。
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df["label"])

# Check that no identical review text appears in multiple partitions.
# 確認沒有相同評論文字同時出現在不同資料組。
assert set(train_df["sentence"]).isdisjoint(val_df["sentence"])

# Check the independence of validation and test reviews.
# 確認驗證集與測試集不包含相同評論。
assert set(val_df["sentence"]).isdisjoint(test_df["sentence"])

# Check the independence of training and test reviews.
# 確認訓練集與測試集不包含相同評論。
assert set(train_df["sentence"]).isdisjoint(test_df["sentence"])

# Load the tokenizer matching the selected pretrained BERT checkpoint.
# 載入與預訓練 BERT 模型相符的 Tokenizer。
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Define one text-processing rule for every partition.
# 為全部資料組定義相同的文字處理規則。
def encode(batch):
    # Tokenize the reviews, truncating long hotel comments.
    # 將飯店評論轉成 Token，並截斷過長文字。
    return tokenizer(batch["sentence"], truncation=True, max_length=128)

# Convert the training partition into a Hugging Face dataset.
# 將訓練資料轉成 Hugging Face Dataset。
train_data = Dataset.from_pandas(train_df, preserve_index=False).map(encode, batched=True)

# Convert the validation partition using the same tokenizer.
# 使用相同 Tokenizer 轉換驗證資料。
val_data = Dataset.from_pandas(val_df, preserve_index=False).map(encode, batched=True)

# Convert the test partition without using it for model selection.
# 轉換測試資料，但不拿來選擇模型。
test_data = Dataset.from_pandas(test_df, preserve_index=False).map(encode, batched=True)

# Display exact partition sizes after stratified splitting.
# 顯示分層切分後三份資料的實際筆數。
print("Train:", len(train_data), "Validation:", len(val_data), "Test:", len(test_data))

# Display label distributions in every partition.
# 顯示各組的正負面數量，確認比例大致一致。
print("各組標籤：", {name: part["label"].value_counts().sort_index().to_dict() for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]})

## 2｜設定訓練與 Checkpoint

| 參數 | 預設 | 作用 |
|---|---|---|
| `EPOCHS` | `3` | 最多訓練 3 輪 |
| `MAX_STEPS` | `-1` | `-1` 依 Epoch；`500` 最多訓練 500 Steps（覆蓋 Epoch 上限） |
| `EVAL_STEPS` | `100` | 每 100 Steps 驗證一次 |
| `SAVE_STEPS` | `EVAL_STEPS` | 驗證時儲存 Checkpoint |
| `LOGGING_STEPS` | `25` | 記錄 Training Loss |
| `BATCH_SIZE` | `16` | 每批 16 筆 |
| `LEARNING_RATE` | `3e-5` | 學習率 |
| `USE_EARLY_STOPPING` | `True` | 啟用提前停止 |
| `EARLY_STOPPING_PATIENCE` | `2` | 連續 2 次驗證未改善就停止 |
| `EARLY_STOPPING_THRESHOLD` | `0.001` | 最低改善幅度 |

**Step 跨 Epoch 累計。** 預設每輪約 **311 Steps**，3 輪上限約 **933 Steps**；最佳模型依最低 Validation Loss 選取。

**注意：**Early Stopping 仍可能提前停止；若不希望提前停止，設 `USE_EARLY_STOPPING = False`。


In [ ]:
# Set the maximum number of complete training epochs.
# 設定訓練最多進行幾個 Epoch。
EPOCHS = 3

# Set an optional step limit; -1 keeps the epoch limit in control.
# 設定選用的 Steps 上限；-1 代表仍由 Epoch 控制訓練。
MAX_STEPS = -1

# Evaluate every N optimizer updates, across all epochs.
# 每累計 N 次 Optimizer 更新就驗證一次，跨 Epoch 持續計算。
EVAL_STEPS = 100

# Save checkpoints at the same step as each validation.
# 在每次驗證的相同步數儲存 Checkpoint。
SAVE_STEPS = EVAL_STEPS

# Record training loss every N optimizer updates.
# 每累計 N 次 Optimizer 更新就記錄 Training Loss。
LOGGING_STEPS = 25

# Enable stopping when validation loss stops improving.
# 啟用 Validation Loss 不再改善時的提前停止功能。
USE_EARLY_STOPPING = True

# Allow two consecutive validations without sufficient improvement.
# 允許連續兩次驗證未達改善標準，之後提前停止。
EARLY_STOPPING_PATIENCE = 2

# Require validation loss to drop by more than this amount.
# Validation Loss 必須下降超過此數值，才視為有效改善。
EARLY_STOPPING_THRESHOLD = 0.001

# Set how many reviews each training batch contains.
# 設定每個訓練 Batch 包含多少筆飯店評論。
BATCH_SIZE = 16

# Set the learning rate used to update model weights.
# 設定更新模型權重時使用的 Learning Rate。
LEARNING_RATE = 3e-5

# Define the project output folder.
# 指定本次訓練的輸出資料夾。
OUTPUT_DIR = Path("03_bert_reviews_output")

# Keep temporary checkpoints in their own folder.
# 將訓練期間的 Checkpoint 集中存放在獨立資料夾。
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

# Reserve a separate folder for the final best model.
# 另外指定最佳模型的獨立資料夾。
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

# Load pretrained Chinese BERT with a two-class classification head.
# 載入預訓練中文 BERT，並建立二元分類頭。
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

# Configure training and select the best model using validation loss.
# 設定訓練流程，並根據 Validation Loss 選擇最佳模型。
settings = TrainingArguments(
    # Save intermediate checkpoints under the checkpoints directory.
    # 將訓練期間的 Checkpoint 儲存到 checkpoints 目錄。
    output_dir=str(CHECKPOINT_DIR),
    # Set the epoch limit, unless a positive step limit overrides it.
    # 設定 Epoch 上限；若 Steps 上限為正整數則由 Steps 優先。
    num_train_epochs=EPOCHS,
    # Choose step-limited training or -1 for epoch-limited training.
    # 設定 Steps 訓練上限，或以 -1 保持 Epoch 訓練上限。
    max_steps=MAX_STEPS,
    # Set the number of examples for each training update batch.
    # 設定每次訓練 Batch 使用的資料筆數。
    per_device_train_batch_size=BATCH_SIZE,
    # Configure a separate batch size for evaluation.
    # 設定驗證階段使用的 Batch Size。
    per_device_eval_batch_size=32,
    # Control optimizer updates through the learning rate.
    # 透過 Learning Rate 控制模型參數更新幅度。
    learning_rate=LEARNING_RATE,
    # Log training loss regularly throughout all epochs.
    # 在所有 Epoch 期間定期記錄 Training Loss。
    logging_strategy="steps",
    # Control how often training loss is logged.
    # 設定 Training Loss 的記錄間隔。
    logging_steps=LOGGING_STEPS,
    # Evaluate validation data by global optimizer steps.
    # 依累計的 Optimizer Step 間隔執行 Validation。
    eval_strategy="steps",
    # Evaluate at every configured step interval.
    # 每隔指定 Step 數進行一次驗證。
    eval_steps=EVAL_STEPS,
    # Save intermediate checkpoints by optimizer steps.
    # 依 Optimizer Step 間隔儲存 Checkpoint。
    save_strategy="steps",
    # Align checkpoint saving with each validation.
    # 讓每一次驗證都對應到可儲存的 Checkpoint。
    save_steps=SAVE_STEPS,
    # Limit disk usage while always retaining the best checkpoint.
    # 限制 Checkpoint 數量，同時保留最佳模型的 Checkpoint。
    save_total_limit=2,
    # Reload the best validated weights after training finishes.
    # 訓練結束後重新載入驗證結果最佳的模型權重。
    load_best_model_at_end=True,
    # Select the checkpoint with the lowest validation loss.
    # 根據最低 Validation Loss 選擇最佳 Checkpoint。
    metric_for_best_model="eval_loss",
    # Treat smaller validation loss as better.
    # 驗證 Loss 越小代表選出的模型越好。
    greater_is_better=False,
    # Enable half precision only when CUDA is available.
    # 只有 CUDA 可用時才啟用半精度訓練。
    fp16=torch.cuda.is_available(),
    # Avoid sending logs to third-party tracking services.
    # 不將訓練紀錄送往第三方追蹤服務。
    report_to="none",
)

# Add early stopping only when it is enabled.
# 只有啟用提前停止時，才建立 Early Stopping Callback。
early_stopping_callbacks = [EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE, early_stopping_threshold=EARLY_STOPPING_THRESHOLD)] if USE_EARLY_STOPPING else []

# Create the trainer with optional early stopping and dynamic padding.
# 建立 Trainer，加入選用的提前停止功能與動態 Padding。
trainer = Trainer(
    # Provide the pretrained classifier.
    # 指定要微調的分類模型。
    model=model,
    # Apply the training configuration.
    # 使用前面設定的訓練參數。
    args=settings,
    # Provide the training split.
    # 傳入訓練資料集。
    train_dataset=train_data,
    # Use validation data to decide when to stop.
    # 使用驗證資料判斷何時提前停止。
    eval_dataset=val_data,
    # Pad sequences to the longest sample within each batch.
    # 根據每個 Batch 的最長句子動態補齊。
    data_collator=DataCollatorWithPadding(tokenizer),
    # Preserve the tokenizer for saving and preprocessing.
    # 保留 Tokenizer 以供儲存及前處理使用。
    processing_class=tokenizer,
    # Enable or disable early stopping without rewriting Trainer.
    # 不必修改 Trainer 就能切換是否提前停止。
    callbacks=early_stopping_callbacks,
)

# Fine-tune the BERT encoder and classification head.
# 微調 BERT Encoder 與二元分類頭。
trainer.train()

# Ensure a best checkpoint was selected through validation.
# 確認已經透過驗證資料選出最佳 Checkpoint。
if trainer.state.best_model_checkpoint is None:
    # Stop rather than mistakenly saving an unvalidated model.
    # 若未完成驗證，不應將未驗證模型誤存為最佳模型。
    raise RuntimeError("沒有找到最佳 Checkpoint，請檢查驗證與儲存設定。")

# Save the reloaded best model in a dedicated folder.
# 將重新載入的最佳模型獨立儲存到 best_model 資料夾。
trainer.save_model(str(BEST_MODEL_DIR))

# Save the matching tokenizer alongside the best model.
# 將對應的 Tokenizer 一起儲存到 best_model 資料夾。
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

# Show the actual source checkpoint used to select the best model.
# 顯示選出最佳模型時使用的原始 Checkpoint 路徑。
print("最佳 Checkpoint：", trainer.state.best_model_checkpoint)

# Show the validation loss of the selected checkpoint.
# 顯示最佳 Checkpoint 對應的 Validation Loss。
print("最佳 Validation Loss：", trainer.state.best_metric)

# Report how many optimizer updates were actually completed.
# 顯示訓練實際執行的 Optimizer 更新次數與原定上限。
print("實際 Step／原定上限：", trainer.state.global_step, "/", trainer.state.max_steps)

# Report which optimizer steps were actually validated.
# 列出實際完成 Validation 的 Optimizer Step。
validated_steps = [event["step"] for event in trainer.state.log_history if "eval_loss" in event]

# Show the exact evaluation checkpoints reached before training ended.
# 顯示訓練停止前實際執行的驗證步數。
print("已完成 Validation 的 Steps：", validated_steps)

# Report whether early stopping was enabled for this run.
# 顯示本次訓練是否啟用提前停止。
print("Early Stopping：", "開啟" if USE_EARLY_STOPPING else "關閉")

# Show the folder that can be opened in Colab's file browser.
# 顯示可在 Colab 檔案瀏覽器中開啟的最佳模型目錄。
print("獨立最佳模型資料夾：", BEST_MODEL_DIR.resolve())

# List the files saved for independent model loading.
# 列出最佳模型資料夾裡可供獨立載入的檔案。
for saved_file in sorted(BEST_MODEL_DIR.iterdir()):
    # Print the file name without exposing unrelated local paths.
    # 僅列出儲存檔名，避免輸出無關的本機路徑。
    print("  -", saved_file.name)

## 選用 1｜查看 Loss 曲線

| 項目 | 內容 |
|---|---|
| X 軸 | 累計 Optimizer Step |
| 曲線 | Training Loss、Validation Loss |
| 最佳模型 | `03_bert_reviews_output/best_model/` |


In [ ]:
# Convert the actual Trainer log into a table.
# 將 Trainer 的實際訓練紀錄轉成表格。
history = pd.DataFrame(trainer.state.log_history)

# Select training loss records from the log.
# 從訓練紀錄取出 Training Loss。
train_history = history.dropna(subset=["loss"])

# Select validation loss records from the log.
# 從訓練紀錄取出 Validation Loss。
val_history = history.dropna(subset=["eval_loss"])

# Use the cumulative optimizer step as the shared horizontal axis.
# 使用累計 Optimizer Step 作為兩條曲線共同的橫軸。
x_axis = "step"

# Draw the measured training-loss curve.
# 繪製實際測得的 Training Loss 曲線。
plt.plot(train_history[x_axis], train_history["loss"], marker="o", label="Train Loss")

# Draw the validation-loss curve for comparison.
# 繪製 Validation Loss 曲線供比較。
plt.plot(val_history[x_axis], val_history["eval_loss"], marker="o", label="Validation Loss")

# Label the optimizer update step on the horizontal axis.
# 將橫軸標示為 Optimizer 更新步數。
plt.xlabel("Optimizer Step")

# Label the measured loss on the vertical axis.
# 標示縱軸為實際測得的 Loss。
plt.ylabel("Loss")

# Show the legend for the two curves.
# 顯示兩條曲線的圖例。
plt.legend()

# Display the completed loss chart.
# 顯示完整的 Loss 圖表。
plt.show()

# Report the best checkpoint selected without using the test set.
# 顯示未使用測試集選出的最佳 Checkpoint。
print("最佳 Checkpoint：", trainer.state.best_model_checkpoint)

# Report where the independent best model can be loaded from.
# 顯示可獨立載入最佳模型的資料夾。
print("最佳模型資料夾：", BEST_MODEL_DIR.resolve())

## 選用 2｜Test 與新評論預測

| 項目 | 內容 |
|---|---|
| 評估 | Accuracy、各類 Precision／Recall／F1 |
| 標籤 | `0` 負面、`1` 正面 |
| 預測模型 | 從 `best_model/` 載入 |

**注意：**Test 只在最後評估；正負面筆數不均，請同時查看兩類指標。


In [ ]:
# Predict the reviews that were reserved for the final test.
# 預測先前保留、沒有參與訓練的測試評論。
test_result = trainer.predict(test_data)

# Select the class with the largest output logit.
# 以最高 Logit 對應的類別作為預測結果。
predicted = test_result.predictions.argmax(axis=1)

# Measure accuracy on the untouched test partition.
# 計算最後測試資料的 Accuracy。
print("Test Accuracy：", round(accuracy_score(test_df["label"], predicted), 3))

# Show precision, recall, and F1 for both review labels.
# 顯示負面與正面兩類的 Precision、Recall 與 F1。
print(classification_report(test_df["label"], predicted, labels=[0, 1], target_names=["負面", "正面"], digits=3))

In [ ]:
# Load the exported best model directly from its saved folder.
# 直接從獨立的 best_model 資料夾載入最佳微調模型。
classifier = pipeline("text-classification", model=str(BEST_MODEL_DIR), tokenizer=str(BEST_MODEL_DIR), device=0 if torch.cuda.is_available() else -1)

In [ ]:
# Choose a new hotel review for prediction.
# 選擇一則新的飯店住宿評論進行預測。
text = "房間乾淨又安靜，櫃檯服務很親切，下次還會入住。"

# Copy either example to replace the review above when practicing.
# 練習時可複製下列其中一句，替換上面的評論。
# text = "房間隔音很差，浴室不乾淨，整晚都沒睡好。"
# text = "交通方便，但早餐選擇很少，整體表現普通。"

# Classify the hotel review using the best fine-tuned model.
# 使用最佳微調模型判斷這則飯店評論。
result = classifier(text, truncation=True, max_length=128)[0]

# Show the predicted class and its model score.
# 顯示預測類別與模型分數。
print(text, "→", result["label"], round(result["score"], 3))

## 完成後確認

- **Checkpoint：** `03_bert_reviews_output/checkpoints/`
- **最佳模型：** `03_bert_reviews_output/best_model/`
- **最後步驟：**查看 Test 指標，輸入自己的評論測試。
